# Project Notebook 1: Data Analytics & Vector Pipeline

**Author:** Vansh Goel
**Date:** October 21, 2025

## 1. Introduction

This notebook serves two purposes as required by the assignment:
1.  **Data Ingestion & Preparation:** It loads the raw `products.csv` file, performs critical cleaning, and engineers new features.
2.  **Vector DB Pipeline:** It generates 384-dimensional text embeddings for each product and uploads them to a Pinecone vector database.
3.  **Analytics:** It performs the exploratory data analysis (EDA) needed to build the Analytics dashboard (e.g., finding top brands and categories).

---

In [2]:
import pandas as pd
import os

print(f"Current working directory: {os.getcwd()}")

# The path from 'notebooks/' folder down to 'data/products.csv'
file_path = '../data/products.csv' 

try:
    # --- THIS IS THE FIX ---
    # Use pandas.read_csv() to load the file directly
    df = pd.read_csv(file_path)
    # ---------------------

    print("\n✅ Data loaded successfully!")
    print(f"Total rows (products): {df.shape[0]}")
    print(f"Total columns (features): {df.shape[1]}")
    
    print("\n--- Column Info & Missing Values ---")
    # This command is powerful: it shows data types AND counts non-null values
    print(df.info())
    
    print("\n--- First 5 Rows (Sample) ---")
    # .head() gives us a quick look at the actual data
    print(df.head())

except FileNotFoundError:
    print(f"\n❌ ERROR: File not found.")
    print(f"Tried to load from: {os.path.abspath(file_path)}")
    print("Please make sure 'products.csv' is inside the 'data' folder.")
except UnicodeDecodeError:
    print(f"\n❌ ERROR: UnicodeDecodeError.")
    print("The file is not in 'utf-8' encoding. Trying to fix...")
    try:
        # Try a different common encoding
        df = pd.read_csv(file_path, encoding='latin1')
        print("\n✅ Data loaded successfully with 'latin1' encoding!")
        print(df.info())
    except Exception as e:
        print(f"\n❌ Still failed. Error: {e}")
except Exception as e:
    print(f"\n❌ An unexpected error occurred: {e}")

Current working directory: c:\Users\Asus\Desktop\product-recommendation-app\notebooks

✅ Data loaded successfully!
Total rows (products): 312
Total columns (features): 12

--- Column Info & Missing Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   title               312 non-null    object
 1   brand               312 non-null    object
 2   description         159 non-null    object
 3   price               215 non-null    object
 4   categories          312 non-null    object
 5   images              312 non-null    object
 6   manufacturer        205 non-null    object
 7   package_dimensions  306 non-null    object
 8   country_of_origin   125 non-null    object
 9   material            218 non-null    object
 10  color               265 non-null    object
 11  uniq_id             312 non-null    object
dtypes: object

In [3]:
# --- Step 1: Handle Missing Descriptions ---
# If a description is missing (NaN), fill it with the content from the 'title' column.
df['description'].fillna(df['title'], inplace=True)

print("✅ Missing descriptions filled.")


# --- Step 2: Create a Combined Text Column for Embeddings ---
# We combine the title and description for a richer embedding context.
df['text_for_embedding'] = df['title'] + ". " + df['description']

print("✅ 'text_for_embedding' column created.")


# --- Step 3: Clean the Price Column ---
# Replace non-numeric characters (like '$') with nothing and convert to a numeric type.
# errors='coerce' will turn any values that can't be converted into NaN (Not a Number).
df['price_numeric'] = pd.to_numeric(df['price'].astype(str).str.replace(r'[^0-9\.]', '', regex=True), errors='coerce')

print("✅ 'price' column cleaned and converted to 'price_numeric'.")


# --- Final Check ---
print("\n--- Final Data Check After Cleaning ---")
print(df[['description', 'text_for_embedding', 'price_numeric']].info())

print("\n--- Sample of the New 'text_for_embedding' Column ---")
print(df[['title', 'text_for_embedding']].head())

✅ Missing descriptions filled.
✅ 'text_for_embedding' column created.
✅ 'price' column cleaned and converted to 'price_numeric'.

--- Final Data Check After Cleaning ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   description         312 non-null    object 
 1   text_for_embedding  312 non-null    object 
 2   price_numeric       215 non-null    float64
dtypes: float64(1), object(2)
memory usage: 7.4+ KB
None

--- Sample of the New 'text_for_embedding' Column ---
                                               title  \
0  GOYMFK 1pc Free Standing Shoe Rack, Multi-laye...   
1  subrtex Leather ding Room, Dining Chairs Set o...   
2  Plant Repotting Mat MUYETOL Waterproof Transpl...   
3  Pickleball Doormat, Welcome Doormat Absorbent ...   
4  JOIN IRON Foldable TV Trays for Eating Set of ...   

                                

C:\Users\Asus\AppData\Local\Temp\ipykernel_24940\362866652.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['description'].fillna(df['title'], inplace=True)


## 2. Text Embedding Generation

Here, we use the `sentence-transformers/all-MiniLM-L6-v2` model via LangChain's `HuggingFaceEmbeddings` wrapper. This model is chosen for its excellent balance of speed, size, and performance. It converts our `text_for_embedding` column into 384-dimensional vectors, capturing the semantic meaning of the text for our recommendation engine.

---

In [4]:
import time
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize the embedding model from HuggingFace
# This model runs locally on your machine
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'} # Use CPU
encode_kwargs = {'normalize_embeddings': False}

print("Initializing HuggingFace embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# 2. Get the list of texts to embed
documents = df['text_for_embedding'].tolist()

print(f"Embedding {len(documents)} documents. This may take a minute...")
start_time = time.time()

# 3. Run the embedding process
# This is the step that turns all your text into vectors
vectors = embeddings.embed_documents(documents)

end_time = time.time()
print(f"\n✅ Embedding complete! Took {end_time - start_time:.2f} seconds.")

# 4. Check the results and store in our DataFrame
print(f"Total vectors created: {len(vectors)}")
print(f"Dimensions of each vector: {len(vectors[0])}")

# 5. Add the vectors to our DataFrame
df['vector'] = vectors

print("\n--- DataFrame with 'vector' column ---")
print(df.head())

Initializing HuggingFace embedding model...
Embedding 312 documents. This may take a minute...

✅ Embedding complete! Took 4.41 seconds.
Total vectors created: 312
Dimensions of each vector: 384

--- DataFrame with 'vector' column ---
                                               title            brand  \
0  GOYMFK 1pc Free Standing Shoe Rack, Multi-laye...           GOYMFK   
1  subrtex Leather ding Room, Dining Chairs Set o...          subrtex   
2  Plant Repotting Mat MUYETOL Waterproof Transpl...          MUYETOL   
3  Pickleball Doormat, Welcome Doormat Absorbent ...          VEWETOL   
4  JOIN IRON Foldable TV Trays for Eating Set of ...  JOIN IRON Store   

                                         description   price  \
0  multiple shoes, coats, hats, and other items E...  $24.99   
1                     subrtex Dining chairs Set of 2     NaN   
2  Plant Repotting Mat MUYETOL Waterproof Transpl...   $5.98   
3  The decorative doormat features a subtle textu...  $13.99   
4  Set

## 3. Pinecone Vector Database Upload

After generating the vectors, we connect to our Pinecone index. We structure the data into `(id, vector, metadata)` tuples. The `metadata` (like title, brand, and images) is stored alongside the vector, allowing us to retrieve human-readable data after performing a vector search.

---

In [5]:
import os
import pandas as pd
from pinecone import Pinecone
from dotenv import load_dotenv

# 1. Load environment variables from your backend's .env file
env_path = '../backend/.env'
load_dotenv(dotenv_path=env_path)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_HOST = os.getenv("PINECONE_HOST")

if not PINECONE_API_KEY or not PINECONE_HOST:
    print("❌ Error: PINECONE_API_KEY or PINECONE_HOST not found in .env file.")
    print(f"Please check your file at {os.path.abspath(env_path)}")
else:
    print("✅ Pinecone credentials loaded successfully.")

    # 2. Connect to Pinecone
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index(host=PINECONE_HOST)
    
    print("Connecting to Pinecone index...")

    # 3. Format the data for upload
    to_upsert = []
    for i, row in df.iterrows():
        
        # Create the metadata dict
        metadata = {
            "title": row['title'],
            "brand": row['brand'],
            "categories": row['categories'],
            "images": row['images'],
            "price": row['price_numeric']
        }
        
        # Clean metadata: remove any keys with missing (NaN) values
        cleaned_metadata = {k: v for k, v in metadata.items() if pd.notna(v)}

        to_upsert.append({
            "id": row['uniq_id'],          # The unique string ID
            "values": row['vector'],      # The 384-dim vector
            "metadata": cleaned_metadata  # The clean dictionary of data
        })

    print(f"✅ Data formatted for upsert. Total items: {len(to_upsert)}")

    # 4. Upsert (upload) the data
    print("Uploading vectors to Pinecone...")
    index.upsert(vectors=to_upsert)

    print("\n🎉 UPLOAD COMPLETE! 🎉")
    
    # 5. Check the index stats to confirm
    stats = index.describe_index_stats()
    print("\n--- Index Stats ---")
    print(stats)

✅ Pinecone credentials loaded successfully.
Connecting to Pinecone index...
✅ Data formatted for upsert. Total items: 312
Uploading vectors to Pinecone...

🎉 UPLOAD COMPLETE! 🎉

--- Index Stats ---
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 305}},
 'total_vector_count': 305,
 'vector_type': 'dense'}


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               312 non-null    object 
 1   brand               312 non-null    object 
 2   description         312 non-null    object 
 3   price               215 non-null    object 
 4   categories          312 non-null    object 
 5   images              312 non-null    object 
 6   manufacturer        205 non-null    object 
 7   package_dimensions  306 non-null    object 
 8   country_of_origin   125 non-null    object 
 9   material            218 non-null    object 
 10  color               265 non-null    object 
 11  uniq_id             312 non-null    object 
 12  text_for_embedding  312 non-null    object 
 13  price_numeric       215 non-null    float64
 14  vector              312 non-null    object 
dtypes: float64(1), object(14)
memory usage: 36.7+ KB
